# Xero API — Source Data Profiling

**Project:** finance-analytics-pipeline
**Source:** Xero Accounting API v2.0 — Demo Company (Global)
**Run date:** 2026-08-10

## Purpose

Profile the Xero source data before designing the warehouse model. Specifically:

1. Inventory what each endpoint returns — columns, types, nullability, cardinality
2. Identify primary keys and validate uniqueness
3. Validate foreign key relationships and join cardinality
4. Test business rules that affect measure correctness
5. Document constraints that limit what analysis is defensible
6. Propose a star schema justified by the above

## Reproducibility

This notebook is designed to run top-to-bottom from a fresh kernel
(**Restart & Run All**). It does not use `os.chdir` or depend on cell
execution order beyond the documented flow.

**Prerequisite:** `tokens.json` must exist in the project root — run
`python src/auth.py` first if it doesn't.

---
## 1. Setup & Authentication

In [1]:
import base64
import json
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
import os

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Resolve project root without chdir — works whether the kernel starts
# in notebooks/ or the project root.
CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "tokens.json").exists() else CWD.parent
assert (PROJECT_ROOT / "tokens.json").exists(), (
    f"tokens.json not found from {CWD}. Run `python src/auth.py` first."
)

load_dotenv(PROJECT_ROOT / ".env")
CLIENT_ID = os.getenv("XERO_CLIENT_ID")
CLIENT_SECRET = os.getenv("XERO_CLIENT_SECRET")

TOKEN_PATH = PROJECT_ROOT / "tokens.json"
TOKEN_URL = "https://identity.xero.com/connect/token"
BASE = "https://api.xero.com/api.xro/2.0"
TENANT_NAME = "Demo Company (Global)"

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\klean\Projects\finance_analytics_pipeline


In [2]:
def load_tokens():
    with open(TOKEN_PATH) as f:
        return json.load(f)


def save_tokens(tok):
    with open(TOKEN_PATH, "w") as f:
        json.dump(tok, f, indent=2)


def refresh_tokens():
    """Exchange the refresh token for a new access token.

    Xero rotates refresh tokens: each refresh returns a NEW refresh token
    and invalidates the old one, so the result must be persisted.
    Access tokens live 30 minutes; refresh tokens 60 days.
    """
    tok = load_tokens()
    basic = base64.b64encode(f"{CLIENT_ID}:{CLIENT_SECRET}".encode()).decode()
    r = requests.post(
        TOKEN_URL,
        headers={
            "Authorization": f"Basic {basic}",
            "Content-Type": "application/x-www-form-urlencoded",
        },
        data={"grant_type": "refresh_token", "refresh_token": tok["refresh_token"]},
        timeout=30,
    )
    r.raise_for_status()
    new = r.json()
    tok["access_token"] = new["access_token"]
    tok["refresh_token"] = new["refresh_token"]
    tok["expires_in"] = new["expires_in"]
    save_tokens(tok)
    print("Access token refreshed.")
    return tok


TOKENS = load_tokens()
TENANT = next(c for c in TOKENS["connections"] if c["tenantName"] == TENANT_NAME)
TENANT_ID = TENANT["tenantId"]

print(f"Tenant: {TENANT['tenantName']}")
print(f"Tenant ID: {TENANT_ID}")
print(f"Connections available: {[c['tenantName'] for c in TOKENS['connections']]}")

Tenant: Demo Company (Global)
Tenant ID: 15cceb60-06a5-4237-886f-4bb2532ee2a1
Connections available: ['Demo Company (Global)', 'Xero Analytics Pipeline']


In [3]:
def _headers():
    return {
        "Authorization": f"Bearer {TOKENS['access_token']}",
        "Xero-tenant-id": TENANT_ID,
        "Accept": "application/json",
    }


def get(endpoint, _retry=True, **params):
    """GET a Xero endpoint.

    Handles the two failure modes that interrupt exploration:
      401 -> access token expired, refresh once and retry
      429 -> rate limited (60 calls/min, 5000/day per tenant),
             wait for the Retry-After window and retry
    """
    global TOKENS
    r = requests.get(f"{BASE}/{endpoint}", headers=_headers(), params=params, timeout=30)

    if r.status_code == 401 and _retry:
        TOKENS = refresh_tokens()
        return get(endpoint, _retry=False, **params)

    if r.status_code == 429:
        wait = int(r.headers.get("Retry-After", 60)) + 1
        print(f"Rate limited — sleeping {wait}s")
        time.sleep(wait)
        return get(endpoint, _retry=_retry, **params)

    r.raise_for_status()
    return r.json()


# smoke test
_org = get("Organisation")["Organisations"][0]
print(f"Connected to: {_org['Name']}  |  base currency: {_org.get('BaseCurrency')}")

Connected to: Demo Company (Global)  |  base currency: USD


---
## 2. Endpoint Inventory

Record counts per endpoint. This establishes the scale of the dataset before
committing to a model, and confirms which endpoints our OAuth scopes permit.

In [4]:
ENDPOINTS = [
    "Invoices",
    "Contacts",
    "Accounts",
    "BankTransactions",
    "Payments",
    "CreditNotes",
    "Items",
    "TaxRates",
    "TrackingCategories",
]

inventory = []
for ep in ENDPOINTS:
    try:
        data = get(ep)
        key = next(k for k in data if k not in
                   ("Id", "Status", "ProviderName", "DateTimeUTC"))
        inventory.append({"endpoint": ep, "records": len(data.get(key, [])),
                          "result": "ok"})
    except requests.HTTPError as e:
        code_ = e.response.status_code
        note = "insufficient scope" if code_ == 401 else f"HTTP {code_}"
        inventory.append({"endpoint": ep, "records": None, "result": note})
    time.sleep(0.4)

pd.DataFrame(inventory)

,endpoint,records,result
0,Invoices,68,ok
1,Contacts,50,ok
2,Accounts,58,ok
3,BankTransactions,22,ok
4,Payments,51,ok
5,CreditNotes,5,ok
6,Items,16,ok
7,TaxRates,8,ok
8,TrackingCategories,1,ok


### Findings — endpoint inventory

**All nine endpoints returned data.** No scope failures, despite only four
accounting scopes being requested (`contacts`, `invoices`, `banktransactions`,
`settings`) plus `payments` added later.

| Endpoint | Records | In scope for model? |
|---|---|---|
| Invoices | 68 | Yes — primary fact source |
| Contacts | 50 | Yes — `dim_contact` |
| Accounts | 58 | Yes — `dim_account` |
| Payments | 51 | Yes — `fact_payments` |
| BankTransactions | 22 | Deferred — see §4 note |
| Items | 16 | Optional — `dim_item`, but see §4 |
| CreditNotes | 5 | Deferred — contra-revenue, not modelled v1 |
| TaxRates | 8 | Reference only |
| TrackingCategories | 1 | Not used — no meaningful dimension |

**Notes**

- `CreditNotes`, `Items`, `TaxRates` and `TrackingCategories` were reachable
  under `accounting.settings.read` and the invoice scope — no additional
  consent was needed.
- `/Payments` initially returned **401 with `insufficient_scope`**, not 403.
  Scope was added to `src/auth.py` and re-consented. Worth remembering: Xero
  signals a missing scope as 401, which is easy to misread as an expired token.
- Volumes are small throughout. This constrains what analysis is defensible —
  see §7.

---
## 3. Extraction

### Note on invoice line items

Xero's `GET /Invoices` **list** endpoint returns invoice headers only — the
`LineItems` key is present but empty. Line-level detail requires a separate
`GET /Invoices/{InvoiceID}` call per invoice.

This matters for two reasons:

1. **Grain.** Line-item grain is only reachable via detail calls. Header grain
   would lose the `AccountID` join, and with it any spend-by-category analysis.
2. **Rate limits.** Xero permits 60 calls/minute and 5,000/day per tenant.
   The detail loop below is throttled accordingly.

In [5]:
t0 = time.time()

# --- list-level pulls -------------------------------------------------
invoices_list = get("Invoices")["Invoices"]
contacts      = get("Contacts")["Contacts"]
accounts      = get("Accounts")["Accounts"]
banktxns      = get("BankTransactions")["BankTransactions"]
payments      = get("Payments")["Payments"]

print(f"list pulls: invoices={len(invoices_list)} contacts={len(contacts)} "
      f"accounts={len(accounts)} banktxns={len(banktxns)} payments={len(payments)}")

list pulls: invoices=68 contacts=50 accounts=58 banktxns=22 payments=51


In [6]:
# --- invoice detail pulls (one per invoice, for LineItems) ------------
# Throttled to stay inside the 60 calls/minute limit.
THROTTLE_SEC = 1.05

invoice_details = []
for n, inv in enumerate(invoices_list, 1):
    invoice_details.append(get(f"Invoices/{inv['InvoiceID']}")["Invoices"][0])
    if n % 10 == 0 or n == len(invoices_list):
        print(f"  {n}/{len(invoices_list)} invoices fetched")
    time.sleep(THROTTLE_SEC)

print(f"\nFetched {len(invoice_details)} invoice details "
      f"in {time.time() - t0:.0f}s")

  10/68 invoices fetched
  20/68 invoices fetched
  30/68 invoices fetched
  40/68 invoices fetched
  50/68 invoices fetched
  60/68 invoices fetched
  68/68 invoices fetched

Fetched 68 invoice details in 93s


In [7]:
# --- flatten line items, carrying invoice-level context ---------------
invoice_lines = [
    {
        **li,
        "InvoiceID": d["InvoiceID"],
        "InvoiceNumber": d.get("InvoiceNumber"),
        "InvoiceType": d["Type"],
        "InvoiceStatus": d["Status"],
        "LineAmountTypes": d["LineAmountTypes"],
        "InvoiceDate": d.get("DateString"),
        "ContactID": d.get("Contact", {}).get("ContactID"),
    }
    for d in invoice_details
    for li in d.get("LineItems", [])
]

RAW = {
    "invoices": invoice_details,
    "lines":    invoice_lines,
    "payments": payments,
    "contacts": contacts,
    "accounts": accounts,
    "banktxns": banktxns,
}

# json_normalize flattens nested objects (e.g. Contact.ContactID),
# which is what exposes the foreign keys as columns.
dfs = {k: pd.json_normalize(v) for k, v in RAW.items()}

pd.DataFrame(
    [{"entity": k, "rows": d.shape[0], "cols": d.shape[1]} for k, d in dfs.items()]
)

,entity,rows,cols
0,invoices,68,61
1,lines,77,22
2,payments,51,46
3,contacts,50,32
4,accounts,58,19
5,banktxns,22,25


---
## 4. Schema Profiling

For each entity: column name, dtype, null rate, distinct count, and a sample
value. Sorted by null rate so densely-populated (likely useful) columns appear
first and mostly-empty ones sink to the bottom.

**What to look for:**

- Columns that are 100% null → candidates to drop
- Columns where `n_unique == n_rows` → primary key candidates
- Columns where `n_unique == 1` → constant, no analytical value
- Nested columns (`Foo.Bar`) → foreign keys or objects needing their own table
- `object` dtype on something that should be numeric or a date → needs casting

In [10]:
def _nunique_safe(s):
    """nunique() fails on columns containing lists/dicts (unhashable).
    Fall back to comparing string representations."""
    try:
        return s.nunique(dropna=True)
    except TypeError:
        return s.dropna().astype(str).nunique()


def _kind(s):
    """Flag columns holding nested structures — these stay in the raw
    JSON layer rather than being modelled into dimensions."""
    nn = s.dropna()
    if len(nn) == 0:
        return "empty"
    v = nn.iloc[0]
    if isinstance(v, list):
        return "list"
    if isinstance(v, dict):
        return "dict"
    return "scalar"


def profile(df, name):
    out = pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "kind":     [_kind(df[c]) for c in df.columns],
        "non_null": df.notna().sum(),
        "null_pct": (df.isna().mean() * 100).round(1),
        "n_unique": [_nunique_safe(df[c]) for c in df.columns],
    })
    samples = []
    for c in df.columns:
        s = df[c].dropna()
        samples.append(str(s.iloc[0])[:50] if len(s) else None)
    out["sample"] = samples
    out.index.name = f"{name} ({len(df)} rows x {df.shape[1]} cols)"
    return out.sort_values(["null_pct", "n_unique"], ascending=[True, False])

In [11]:
profile(dfs["invoices"], "INVOICES")

,dtype,kind,non_null,null_pct,n_unique,sample
INVOICES (68 rows x 61 cols),,,,,,
InvoiceID,str,scalar,68,0.0,68,fee88eea-f2aa-4a71-a372-33d6d83d3c45
LineItems,object,list,68,0.0,68,"[{'Description': 'Marketing guides', 'UnitAmount':"
UpdatedDateUTC,str,scalar,68,0.0,68,/Date(1229650679057+0000)/
UpdatedDateUTCString,str,scalar,68,0.0,63,2008-12-19T01:37:59Z
InvoiceNumber,str,scalar,68,0.0,42,INV-0027
SubTotal,float64,scalar,68,0.0,42,365.82
TotalTax,float64,scalar,68,0.0,42,30.18
Total,float64,scalar,68,0.0,42,396.0
DueDateString,str,scalar,68,0.0,38,2026-08-15T00:00:00


**Invoices — notes**

61 columns, of which roughly a dozen are useful. The rest is `json_normalize`
expanding nested objects.

**Keep**

`InvoiceID` (PK) · `InvoiceNumber` · `Type` · `Status` · `LineAmountTypes` ·
`DateString` · `DueDateString` · `SubTotal` · `TotalTax` · `Total` ·
`AmountDue` · `AmountPaid` · `AmountCredited` · `Contact.ContactID` (FK) ·
`Reference` · `RepeatingInvoiceID`

**Drop — constant (`n_unique = 1`), no analytical value**

`CurrencyCode` (USD only) · `IsDiscounted` · `HasErrors` · `HasAttachments` ·
`Attachments` · `Prepayments` · `Overpayments` · `InvoicePaymentServices`

**Drop — denormalised contact payload**

All 20+ `Contact.*` columns beyond `Contact.ContactID`. This is the contact
record embedded in every invoice. It belongs in `dim_contact`, sourced from
`/Contacts`, not carried redundantly on the fact.

**Drop — nested lists handled elsewhere**

`LineItems` (exploded into the lines table) · `Payments` (sourced from
`/Payments`; note it is 50% null here and the nested copy is a summary) ·
`CreditNotes` (5 records, deferred)

**Type issues**

- **Two date representations.** `Date` is Xero's epoch-milliseconds format
  (`/Date(1786233600000+0000)/`), unusable without parsing. `DateString` is
  ISO-8601. **Use the `*String` variants throughout.**
- All monetary fields are already `float64` — no casting needed.

**⚠️ `UpdatedDateUTC` is unreliable in the Demo Company.** Sample value is
`2008-12-19`, while invoice dates are 2026. The demo data carries stale
modification timestamps. This matters because `If-Modified-Since` incremental
extraction depends on it — incremental loading cannot be validated against this
source, only implemented and unit-tested.

In [12]:
profile(dfs["lines"], "INVOICE LINES")

,dtype,kind,non_null,null_pct,n_unique,sample
INVOICE LINES (77 rows x 22 cols),,,,,,
LineItemID,str,scalar,77,0.0,77,8dd05881-be4a-4765-b95e-cce0d7395f9f
InvoiceID,str,scalar,77,0.0,68,fee88eea-f2aa-4a71-a372-33d6d83d3c45
Description,str,scalar,77,0.0,42,Marketing guides
TaxAmount,float64,scalar,77,0.0,42,30.18
InvoiceNumber,str,scalar,77,0.0,42,INV-0027
LineAmount,float64,scalar,77,0.0,41,396.0
UnitAmount,float64,scalar,77,0.0,39,99.0
InvoiceDate,str,scalar,77,0.0,35,2026-08-09T00:00:00
ContactID,str,scalar,77,0.0,29,5b96e86b-418e-48e8-8949-308c14aec278


**Invoice lines — notes**

22 columns, nearly all useful. This is the cleanest entity in the source.

**Keep**

`LineItemID` (PK) · `InvoiceID` (FK) · `AccountID` (FK) · `AccountCode` ·
`Description` · `Quantity` · `UnitAmount` · `LineAmount` · `TaxAmount` ·
`TaxType` · plus the invoice context carried during flattening
(`InvoiceType`, `InvoiceStatus`, `LineAmountTypes`, `InvoiceDate`, `ContactID`)

**Drop**

`ValidationErrors` (empty for all rows) · `Tracking` (nested list; only 4
distinct values, 1 tracking category exists — not worth a dimension)

**Cardinality and coverage**

- 77 lines across 68 invoices — **1.13 lines per invoice**, max 3.
  The dataset is wide, not deep.
- `AccountID` has **17 distinct values** against 58 accounts in the chart of
  accounts. Only 29% of accounts are ever transacted against. Any
  spend-by-category breakdown has 17 buckets, not 58.
- `Quantity` has only 6 distinct values — near-categorical in practice.

**`Item` linkage is partial.** `ItemCode` / `Item.ItemID` are populated on
**30 of 77 lines (39%)**, covering 8 distinct items out of the 16 in
`/Items`. A `dim_item` would leave 61% of lines unattributed. **Recommendation:
skip `dim_item` in v1** and use `dim_account` for categorisation, which has
100% coverage.

In [13]:
profile(dfs["payments"], "PAYMENTS")

,dtype,kind,non_null,null_pct,n_unique,sample
PAYMENTS (51 rows x 46 cols),,,,,,
PaymentID,str,scalar,51,0.0,51,655ad293-0b35-4c2c-9a97-0ee979dd365f
UpdatedDateUTC,str,scalar,51,0.0,44,/Date(1229803587127+0000)/
Invoice.InvoiceID,str,scalar,51,0.0,35,cb5119d0-9759-49d3-800b-7d0a90818178
UpdatedDateUTCString,str,scalar,51,0.0,32,2008-12-20T20:06:27Z
Invoice.InvoiceNumber,str,scalar,51,0.0,27,INV-0001
BankAmount,float64,scalar,51,0.0,23,541.25
Amount,float64,scalar,51,0.0,23,541.25
Invoice.Contact.ContactID,str,scalar,51,0.0,23,fd89489e-699c-4d77-a881-10c127bfbeb3
Invoice.Contact.Name,str,scalar,51,0.0,23,Hamilton Smith Ltd


**Payments — notes**

46 columns, of which 7 matter. This is the most heavily denormalised entity —
the entire invoice *and* its contact are embedded in every payment record.

**Keep**

`PaymentID` (PK) · `Invoice.InvoiceID` (FK) · `Account.AccountID` (FK) ·
`Amount` · `BankAmount` · `Date`/`DateString` · `PaymentType` · `Status` ·
`IsReconciled` · `BatchPaymentID`

**Drop — denormalised**

All `Invoice.*` and `Invoice.Contact.*` columns beyond `Invoice.InvoiceID`.
Note `Invoice.LineItems` is an empty list for all 51 rows, confirming the
embedded invoice is a summary only.

**Drop — constant**

`HasAccount` · `HasValidationErrors` · `Account.Code` · `Account.CurrencyCode`

**Observations**

- **`Account.AccountID` has exactly 1 distinct value.** Every payment hits the
  same bank account. The `payment → account` join is constant and analytically
  worthless — keep the FK for completeness, but it supports no breakdown.
- **`CurrencyRate` has 2 distinct values including `0.0`.** A rate of zero is
  nonsensical. Since the org is single-currency USD, the field is unused —
  drop rather than attempt to interpret.
- **Batch payments exist:** `BatchPaymentID` populated on 9 of 51 payments
  across 7 distinct batches. Not modelled in v1, but noted — batching means one
  bank movement can settle several invoices.
- **`Status` has 2 values.** See §6.2 — this is the critical finding.

In [14]:
profile(dfs["contacts"], "CONTACTS")

,dtype,kind,non_null,null_pct,n_unique,sample
CONTACTS (50 rows x 32 cols),,,,,,
ContactID,str,scalar,50,0.0,50,9a777d01-2bfb-4623-807d-129d3f077e21
Name,str,scalar,50,0.0,49,Coco Cafe
Phones,object,list,50,0.0,15,"[{'PhoneType': 'DDI'}, {'PhoneType': 'DEFAULT'}, {"
Addresses,object,list,50,0.0,14,"[{'AddressType': 'POBOX'}, {'AddressType': 'STREET"
UpdatedDateUTC,str,scalar,50,0.0,10,/Date(1786385021873+0000)/
IsSupplier,bool,scalar,50,0.0,2,False
IsCustomer,bool,scalar,50,0.0,2,False
ContactStatus,str,scalar,50,0.0,1,ACTIVE
ContactGroups,object,list,50,0.0,1,[]


**Contacts — notes**

**Keep**

`ContactID` (PK) · `Name` · `IsSupplier` · `IsCustomer` · `EmailAddress` ·
`FirstName` · `LastName` · `ContactStatus`

**Drop — constant**

`ContactStatus` is `ACTIVE` for all 50 — no archived contacts, so no filter is
needed (unlike the transactional entities). Keep it documented, drop it from
the dimension.

**Drop — nested**

`Addresses`, `Phones`, `ContactGroups`, `ContactPersons` are nested arrays that
flatten badly. A contact can hold several addresses of different types
(`POBOX`, `STREET`). Modelling them properly needs a bridge table; the
analytical value here is nil. **Leave in the raw JSON layer.**

**⚠️ Do not model `Balances.*` into `dim_contact`.**
`Balances.AccountsPayable.Outstanding`, `.Overdue`, and the receivable
equivalents are populated on 16 of 50 contacts. These are **API-computed
aggregates as of request time**, not source facts. Putting them in a dimension
would freeze a point-in-time snapshot that silently goes stale and can
contradict the facts derived from invoices. Outstanding balances must be
calculated in the mart layer from `fact_invoice_lines` and `fact_payments`.

**⚠️ `Name` is not unique — 49 distinct across 50 rows.** Two contacts share a
name. Any join or grouping on contact must use `ContactID`. Worth a dbt
uniqueness test on the key and an explicit note that name is not a business key.

**Coverage:** only **29 of 50 contacts** appear on any invoice. 21 will be
orphan dimension rows with no facts — legitimate, but the dashboard should
default to filtering them out.

In [15]:
profile(dfs["accounts"], "ACCOUNTS")

,dtype,kind,non_null,null_pct,n_unique,sample
ACCOUNTS (58 rows x 19 cols),,,,,,
AccountID,str,scalar,58,0.0,58,562555f2-8cde-4ce9-8203-0363922537a4
Code,str,scalar,58,0.0,58,090
Name,str,scalar,58,0.0,58,Business Bank Account
ReportingCodeName,str,scalar,58,0.0,39,Asset
ReportingCode,str,scalar,58,0.0,38,ASS
Type,str,scalar,58,0.0,10,BANK
Class,str,scalar,58,0.0,5,ASSET
TaxType,str,scalar,58,0.0,3,NONE
BankAccountType,str,scalar,58,0.0,3,BANK


**Accounts — notes**

The cleanest dimension source: 19 columns, no nesting, no nulls on the
important fields.

**Keep**

`AccountID` (PK) · `Code` · `Name` · `Type` · `Class` · `Description` ·
`ReportingCode` · `ReportingCodeName` · `TaxType` · `BankAccountType` ·
`SystemAccount`

**Drop**

`Status` (all `ACTIVE`) · `HasAttachments` (constant) · `AddToWatchlist` ·
`EnablePaymentsToAccount` · `ShowInExpenseClaims` — UI settings, not analysis
attributes.

`BankAccountNumber` and `CurrencyCode` are 96.6% null — populated only on the
2 bank accounts. Keep `BankAccountNumber` out of the warehouse entirely: it is
account-number data with no analytical use.

**`Class` and `Type` are the categorisation backbone** — see the distribution
in the next cell. `Class` gives the 5-way P&L/balance-sheet split;
`ReportingCodeName` (39 distinct) offers a finer grouping if needed.

**Coverage caveat:** 58 accounts exist, but only **17 are referenced by any
invoice line**. `dim_account` will be 71% unused rows.

In [16]:
print("Account Class:")
print(dfs["accounts"]["Class"].value_counts(dropna=False), "\n")
print("Account Type:")
print(dfs["accounts"]["Type"].value_counts(dropna=False), "\n")
print("Account Status:")
print(dfs["accounts"]["Status"].value_counts(dropna=False))

Account Class:
Class
EXPENSE      29
LIABILITY    15
ASSET         9
REVENUE       3
EQUITY        2
Name: count, dtype: int64 

Account Type:
Type
EXPENSE        27
CURRLIAB       14
FIXED           4
REVENUE         3
BANK            2
DIRECTCOSTS     2
CURRENT         2
EQUITY          2
INVENTORY       1
TERMLIAB        1
Name: count, dtype: int64 

Account Status:
Status
ACTIVE    58
Name: count, dtype: int64


In [17]:
profile(dfs["banktxns"], "BANK TRANSACTIONS")

,dtype,kind,non_null,null_pct,n_unique,sample
BANK TRANSACTIONS (22 rows x 25 cols),,,,,,
BankTransactionID,str,scalar,22,0.0,22,b9d7c42c-506a-4e56-a8a1-a41330580834
UpdatedDateUTC,str,scalar,22,0.0,22,/Date(1229813455090+0000)/
DateString,str,scalar,22,0.0,15,2026-05-30T00:00:00
Date,str,scalar,22,0.0,15,/Date(1780099200000+0000)/
SubTotal,float64,scalar,22,0.0,12,15.0
Total,float64,scalar,22,0.0,12,15.0
Contact.ContactID,str,scalar,22,0.0,10,43d1337e-4360-4589-9a76-1d0538c4ce6f
Contact.Name,str,scalar,22,0.0,10,Ridgeway Bank
TotalTax,float64,scalar,22,0.0,9,0.0


**Bank transactions — notes**

**Deferred from the v1 model.** Reasons:

1. **`LineItems` is an empty list for all 22 rows.** Same pattern as invoices —
   line detail requires per-transaction detail calls. Not yet fetched.
2. **`Type` is `SPEND` for all 22.** No `RECEIVE` transactions, so this cannot
   show cash in vs cash out.
3. **`BankAccount.AccountID` has 1 distinct value** — single account, no
   breakdown possible.
4. **8 of 22 are `DELETED`** (36%) — the highest deletion rate of any entity.
   Only 14 live records.

14 usable single-account outflow records adds little beyond what
`fact_payments` already covers. Revisit if the model needs a true cash view;
it would require the detail-call loop and a decision on how bank transactions
relate to invoice payments (they may double-count the same cash movement).

**If modelled later, keep:** `BankTransactionID` (PK) ·
`Contact.ContactID` (FK) · `BankAccount.AccountID` (FK) · `DateString` ·
`Type` · `Status` · `IsReconciled` · `SubTotal` · `TotalTax` · `Total` ·
`Reference`

---
## 5. Key Integrity

### 5.1 Primary keys

A valid PK must be **unique** and **non-null** across all rows.

In [18]:
PK_CANDIDATES = {
    "invoices": "InvoiceID",
    "lines":    "LineItemID",
    "payments": "PaymentID",
    "contacts": "ContactID",
    "accounts": "AccountID",
    "banktxns": "BankTransactionID",
}

pk_rows = []
for entity, col in PK_CANDIDATES.items():
    df = dfs[entity]
    if col not in df.columns:
        pk_rows.append({"entity": entity, "pk": col, "rows": len(df),
                        "unique": None, "nulls": None, "verdict": "COLUMN MISSING"})
        continue
    rows, uniq, nulls = len(df), df[col].nunique(), int(df[col].isna().sum())
    pk_rows.append({
        "entity": entity, "pk": col, "rows": rows, "unique": uniq, "nulls": nulls,
        "verdict": "OK" if (uniq == rows and nulls == 0) else "FAIL",
    })

pd.DataFrame(pk_rows)

,entity,pk,rows,unique,nulls,verdict
0,invoices,InvoiceID,68,68,0,OK
1,lines,LineItemID,77,77,0,OK
2,payments,PaymentID,51,51,0,OK
3,contacts,ContactID,50,50,0,OK
4,accounts,AccountID,58,58,0,OK
5,banktxns,BankTransactionID,22,22,0,OK


### 5.2 Foreign keys & referential integrity

For each relationship: how many child values exist, how many are distinct,
how many are null, and how many point at a parent that doesn't exist (orphans).

**Orphans matter** because they silently drop rows in an inner join. Nulls
matter because they need an explicit "unknown" member in the dimension rather
than being lost.

In [20]:
def fk_check(child, child_col, parent, parent_col, label):
    cdf, pdf = dfs[child], dfs[parent]
    if child_col not in cdf.columns:
        return {"relationship": label, "verdict": "CHILD COL MISSING"}
    vals = cdf[child_col]
    non_null = vals.dropna()
    parent_keys = set(pdf[parent_col].dropna())
    orphans = int((~non_null.isin(parent_keys)).sum())
    nulls = int(vals.isna().sum())
    return {
        "relationship": label,
        "rows": len(cdf),
        "nulls": nulls,
        "distinct": non_null.nunique(),
        "orphans": orphans,
        "verdict": "OK" if orphans == 0 and nulls == 0
                   else ("NULLS" if orphans == 0 else "ORPHANS"),
    }


fk_rows = [
    fk_check("invoices", "Contact.ContactID", "contacts", "ContactID",
             "invoice -> contact"),
    fk_check("lines", "InvoiceID", "invoices", "InvoiceID",
             "line -> invoice"),
    fk_check("lines", "AccountID", "accounts", "AccountID",
             "line -> account"),
    fk_check("lines", "ContactID", "contacts", "ContactID",
             "line -> contact (denormalised)"),
    fk_check("payments", "Invoice.InvoiceID", "invoices", "InvoiceID",
             "payment -> invoice"),
    fk_check("payments", "Account.AccountID", "accounts", "AccountID",
             "payment -> account"),
    fk_check("banktxns", "Contact.ContactID", "contacts", "ContactID",
             "banktxn -> contact"),
    fk_check("banktxns", "BankAccount.AccountID", "accounts", "AccountID",
             "banktxn -> bank account"),
]

pd.DataFrame(fk_rows)

,relationship,rows,nulls,distinct,orphans,verdict
0,invoice -> contact,68,0,29,0,OK
1,line -> invoice,77,0,68,0,OK
2,line -> account,77,0,17,0,OK
3,line -> contact (denormalised),77,0,29,0,OK
4,payment -> invoice,51,0,35,0,OK
5,payment -> account,51,0,1,0,OK
6,banktxn -> contact,22,0,10,0,OK
7,banktxn -> bank account,22,0,1,0,OK


### 5.3 Join cardinality

Uniqueness tells us whether a relationship is 1:1, 1:many or many:1 — which
determines whether a join can fan out and inflate measures.

In [21]:
card = []

# lines per invoice
lpi = dfs["lines"].groupby("InvoiceID").size()
card.append({"relationship": "invoice -> lines",
             "parents": dfs["invoices"]["InvoiceID"].nunique(),
             "children": len(dfs["lines"]),
             "max_per_parent": int(lpi.max()),
             "mean_per_parent": round(lpi.mean(), 2),
             "cardinality": "1:1" if lpi.max() == 1 else "1:M"})

# payments per invoice
ppi = dfs["payments"].groupby("Invoice.InvoiceID").size()
card.append({"relationship": "invoice -> payments",
             "parents": dfs["payments"]["Invoice.InvoiceID"].nunique(),
             "children": len(dfs["payments"]),
             "max_per_parent": int(ppi.max()),
             "mean_per_parent": round(ppi.mean(), 2),
             "cardinality": "1:1" if ppi.max() == 1 else "1:M"})

# invoices per contact
ipc = dfs["invoices"].groupby("Contact.ContactID").size()
card.append({"relationship": "contact -> invoices",
             "parents": dfs["invoices"]["Contact.ContactID"].nunique(),
             "children": len(dfs["invoices"]),
             "max_per_parent": int(ipc.max()),
             "mean_per_parent": round(ipc.mean(), 2),
             "cardinality": "1:1" if ipc.max() == 1 else "1:M"})

pd.DataFrame(card)

,relationship,parents,children,max_per_parent,mean_per_parent,cardinality
0,invoice -> lines,68,77,3,1.13,1:M
1,invoice -> payments,35,51,2,1.46,1:M
2,contact -> invoices,29,68,6,2.34,1:M


### Findings — key integrity

**Primary keys: all six pass.** Unique and non-null across every entity.

| Entity | PK | Rows | Unique | Nulls |
|---|---|---|---|---|
| invoices | `InvoiceID` | 68 | 68 | 0 |
| lines | `LineItemID` | 77 | 77 | 0 |
| payments | `PaymentID` | 51 | 51 | 0 |
| contacts | `ContactID` | 50 | 50 | 0 |
| accounts | `AccountID` | 58 | 58 | 0 |
| banktxns | `BankTransactionID` | 22 | 22 | 0 |

**Referential integrity: zero orphans and zero nulls across all eight
relationships.** Every FK resolves to an existing parent.

Two consequences:

- **No "Unknown" dimension members are required.** Unusual for a real source
  system; a production warehouse would normally need them.
- Inner joins are safe — no rows will silently disappear.

**Cardinality**

| Relationship | Parents | Children | Max/parent | Mean | Type |
|---|---|---|---|---|---|
| invoice → lines | 68 | 77 | 3 | 1.13 | 1:M |
| invoice → payments | 35 | 51 | 2 | 1.46 | 1:M |
| contact → invoices | 29 | 68 | 6 | 2.34 | 1:M |

**Fan-out risk.** All three are one-to-many, so joining invoice headers to
lines repeats header-level values. **Header measures (`Total`, `SubTotal`,
`AmountPaid`) must never be summed at line grain** — they must be aggregated
from a distinct invoice set or held in a separate header-grain model.

**`invoice → payments` is provisional.** See §6.2: after filtering deleted
payments this collapses to **1:1**, and the max-2-per-invoice pattern turns out
to be an artifact rather than genuine partial payment.

**Distinct-value coverage worth noting**

- `invoice → contact`: 29 distinct — 21 of 50 contacts unused
- `line → account`: 17 distinct — 41 of 58 accounts unused
- `payment → account`: **1 distinct** — join is constant, supports no breakdown
- `banktxn → bank account`: 1 distinct — same

---
## 6. Business Rule Validation

Structural checks confirm the data *joins*. These checks confirm it *means*
what we think it means — the errors that produce plausible-looking but wrong
numbers.

### 6.1 Tax semantics: `LineAmountTypes`

Xero invoices are either **Exclusive** (line amounts exclude tax) or
**Inclusive** (line amounts include tax). Summing `LineAmount` across both
without normalising adds inconsistent measures.

Hypothesis: Exclusive lines reconcile to the header `SubTotal`;
Inclusive lines reconcile to `Total`.

In [22]:
recon = []
for d in invoice_details:
    lis = d.get("LineItems", [])
    s = round(sum(li.get("LineAmount", 0) for li in lis), 2)
    recon.append({
        "InvoiceID": d["InvoiceID"],
        "InvoiceNumber": d.get("InvoiceNumber"),
        "LineAmountTypes": d["LineAmountTypes"],
        "n_lines": len(lis),
        "sum_lines": s,
        "SubTotal": d["SubTotal"],
        "TotalTax": d["TotalTax"],
        "Total": d["Total"],
        "matches_subtotal": abs(s - d["SubTotal"]) < 0.01,
        "matches_total": abs(s - d["Total"]) < 0.01,
    })

df_recon = pd.DataFrame(recon)

summary = df_recon.groupby("LineAmountTypes").agg(
    invoices=("InvoiceID", "count"),
    matches_subtotal=("matches_subtotal", "sum"),
    matches_total=("matches_total", "sum"),
)
print(summary, "\n")
print("Invoices matching NEITHER subtotal nor total:")
print(df_recon[~df_recon.matches_subtotal & ~df_recon.matches_total]
      [["InvoiceNumber", "LineAmountTypes", "sum_lines", "SubTotal", "Total"]])

                 invoices  matches_subtotal  matches_total
LineAmountTypes                                           
Exclusive              49                49              1
Inclusive              19                 0             19 

Invoices matching NEITHER subtotal nor total:
Empty DataFrame
Columns: [InvoiceNumber, LineAmountTypes, sum_lines, SubTotal, Total]
Index: []


In [23]:
# Proposed normalisation, tested against the header values.
def net_amount(row_lines, line_amount_types):
    """Return tax-exclusive total for a set of lines."""
    if line_amount_types == "Exclusive":
        return sum(li.get("LineAmount", 0) for li in row_lines)
    # Inclusive: LineAmount already contains tax
    return sum(li.get("LineAmount", 0) - li.get("TaxAmount", 0) for li in row_lines)


check = []
for d in invoice_details:
    lis = d.get("LineItems", [])
    net = round(net_amount(lis, d["LineAmountTypes"]), 2)
    check.append({
        "LineAmountTypes": d["LineAmountTypes"],
        "net": net,
        "SubTotal": d["SubTotal"],
        "ok": abs(net - d["SubTotal"]) < 0.02,
    })

dfc = pd.DataFrame(check)
print("Normalised net vs SubTotal:")
print(dfc.groupby("LineAmountTypes")["ok"].agg(["sum", "count"]))
print("\nFailures:")
print(dfc[~dfc.ok])

Normalised net vs SubTotal:
                 sum  count
LineAmountTypes            
Exclusive         49     49
Inclusive         19     19

Failures:
Empty DataFrame
Columns: [LineAmountTypes, net, SubTotal, ok]
Index: []


**Finding — tax semantics: rule confirmed, normalisation validated**

**The hypothesis holds exactly, with no exceptions:**

| `LineAmountTypes` | Invoices | Matches `SubTotal` | Matches `Total` |
|---|---|---|---|
| Exclusive | 49 | **49** | 1 |
| Inclusive | 19 | 0 | **19** |

Zero invoices matched neither. The single Exclusive invoice matching both is a
zero-tax record (`TaxType: NONE`, a deposit) where `SubTotal == Total`, so both
tests pass trivially — not a contradiction.

**Why this matters.** `LineAmount` means two different things depending on the
invoice. Summing across both without normalising adds tax-exclusive and
tax-inclusive amounts together — the equivalent of summing a column where some
values are metres and some are feet. The result looks entirely plausible and is
simply wrong, overstating by the tax on 19 of 68 invoices.

**Normalisation logic for staging — validated 68/68, zero failures:**

```sql
case
    when line_amount_types = 'Exclusive' then line_amount
    when line_amount_types = 'Inclusive' then line_amount - tax_amount
end as line_amount_net,

case
    when line_amount_types = 'Exclusive' then line_amount + tax_amount
    when line_amount_types = 'Inclusive' then line_amount
end as line_amount_gross
```

Both Exclusive (49/49) and Inclusive (19/19) reconcile to header `SubTotal`
after normalisation.

**dbt test to write**

Assert `sum(line_amount_net) = invoice.subtotal` per invoice (tolerance 0.02
for float rounding). This test **fails on the raw data and passes on the
modelled data** — it encodes the finding rather than merely restating it.

### 6.2 Payments: partial payments and status alignment

More payments than PAID invoices implies either multiple payments per invoice
(partial payment) or payments against invoices not marked PAID. Either changes
whether `fact_payments -> fact_invoices` is safely many:1.

In [24]:
pay_counts = dfs["payments"].groupby("Invoice.InvoiceID").size()
print(f"Payments: {len(dfs['payments'])}")
print(f"Distinct invoices paid: {len(pay_counts)}")
print(f"Invoices with >1 payment: {(pay_counts > 1).sum()}")
print(f"Max payments on one invoice: {pay_counts.max()}\n")

status_map = {d["InvoiceID"]: d["Status"] for d in invoice_details}
type_map = {d["InvoiceID"]: d["Type"] for d in invoice_details}

print("Status of invoices that have payments:")
print(Counter(status_map.get(i) for i in pay_counts.index), "\n")
print("Type of invoices that have payments:")
print(Counter(type_map.get(i) for i in pay_counts.index), "\n")

# do payment amounts reconcile to invoice AmountPaid?
paid_sum = dfs["payments"].groupby("Invoice.InvoiceID")["Amount"].sum()
amt_paid = {d["InvoiceID"]: d.get("AmountPaid", 0) for d in invoice_details}
mismatch = [
    (i, round(v, 2), amt_paid.get(i))
    for i, v in paid_sum.items()
    if abs(v - amt_paid.get(i, 0)) > 0.01
]
print(f"Invoices where sum(payments) != AmountPaid: {len(mismatch)}")
for m in mismatch[:10]:
    print(" ", m)

Payments: 51
Distinct invoices paid: 35
Invoices with >1 payment: 16
Max payments on one invoice: 2

Status of invoices that have payments:
Counter({'PAID': 32, 'AUTHORISED': 3}) 

Type of invoices that have payments:
Counter({'ACCPAY': 19, 'ACCREC': 16}) 

Invoices where sum(payments) != AmountPaid: 17
  ('11e353a8-73f8-4f50-82e0-ff1a2ff8ffba', 2165.0, 1082.5)
  ('12d3bc5a-efe2-4093-b0e3-f6474ef7596d', 11907.5, 5953.75)
  ('1371f05a-4cd3-4760-99a9-e4cf8d98e7bb', 1082.5, 541.25)
  ('1ee8363f-097f-4fb3-a452-8068b947e75e', 1082.5, 541.25)
  ('2a578929-58bf-4eca-90db-1356383aeecb', 3000.0, 1500.0)
  ('2c916a34-14fe-4b38-af86-6b0fcc371a89', 433.0, 216.5)
  ('361a4980-7f53-4b68-820c-f6487f613284', 822.7, 411.35)
  ('4286e4ee-cc4a-4d74-9568-10935b3600bb', 1136.62, 568.31)
  ('463042ef-4343-4a5a-abb8-9a1a3e2e1057', 2814.5, 1407.25)
  ('47b97878-df82-4330-b731-80610cce65d1', 271.7, 135.85)


**Finding — payments: 17 soft-deleted duplicates, cash overstated 45%**

**The signal.** 51 payments against 34 PAID invoices. 16 invoices carried
exactly 2 payments. In all 17 mismatching cases, `sum(payments)` was **exactly
double** `AmountPaid` — 2165.00 vs 1082.50, 11907.50 vs 5953.75, 3000.00 vs
1500.00.

Partial payments would sum *to* `AmountPaid`, not double it. Two payments each
equal to the full amount is a duplicate, not an instalment.

**The cause.** `Status` breaks down as **34 AUTHORISED / 17 DELETED**. Each
doubled invoice has one AUTHORISED and one DELETED payment with the *same date,
same amount, same type*, and `IsReconciled` `True` on the live record,
`False` on the deleted one.

**The Xero API returns soft-deleted payments alongside live ones.** There is no
flag distinguishing them other than `Status`.

**The fix, verified:** filtering to `Status = 'AUTHORISED'` reduces
reconciliation mismatches **from 17 to 0**. Every remaining payment ties exactly
to its invoice's `AmountPaid`.

**Impact if missed:** cash received overstated by roughly 45%. Nothing in the
output would look obviously wrong — the totals are plausible, just doubled.
This is the single most consequential finding in this notebook.

**Cardinality changes as a result:**

| | Before filter | After filter |
|---|---|---|
| Payments | 51 | 34 |
| Distinct invoices paid | 35 | 35 |
| Invoices with >1 payment | 16 | **0** |
| `payment → invoice` | 1:M | **1:1** |

No genuine partial payments exist in this dataset. The 3 payments that appeared
to sit against `AUTHORISED` rather than `PAID` invoices were the deleted ones.

**This is not a payments-only quirk.** The same soft-delete behaviour appears
across transactional entities:

| Entity | Total | Live | Deleted/Voided |
|---|---|---|---|
| Invoices | 68 | 57 | 11 (16%) |
| Payments | 51 | 34 | 17 (33%) |
| Bank transactions | 22 | 14 | 8 (36%) |
| Contacts | 50 | 50 | 0 |
| Accounts | 58 | 58 | 0 |

**General rule for staging:** every transactional model filters to live status.
Dimensions (contacts, accounts) need no filter.

**dbt tests to write**

1. `sum(payment.amount) = invoice.amount_paid` per invoice
2. **`invoice_id` is unique in `fact_payments`** — encodes the 1:1 that only
   holds after filtering. If the filter ever breaks, this fails immediately.

### 6.3 Status and lifecycle

VOIDED and DELETED invoices are not live transactions and must not land in a
fact table as if they were. DRAFT invoices are not yet committed.

In [25]:
inv = dfs["invoices"]
print("Status x Type:")
print(pd.crosstab(inv["Status"], inv["Type"], margins=True), "\n")

excl = inv[inv["Status"].isin(["VOIDED", "DELETED"])]
print(f"VOIDED/DELETED: {len(excl)} of {len(inv)} "
      f"({len(excl)/len(inv)*100:.0f}%)")
print(f"Their total value: {excl['Total'].sum():,.2f}")
print(f"DRAFT: {(inv['Status'] == 'DRAFT').sum()}")

lines_excluded = dfs["lines"][
    dfs["lines"]["InvoiceStatus"].isin(["VOIDED", "DELETED", "DRAFT"])
]
print(f"\nLine items attached to excluded invoices: {len(lines_excluded)} "
      f"of {len(dfs['lines'])}")

Status x Type:
Type        ACCPAY  ACCREC  All
Status                         
AUTHORISED      12       9   21
DELETED          4       0    4
DRAFT            0       2    2
PAID            18      16   34
VOIDED           7       0    7
All             41      27   68 

VOIDED/DELETED: 11 of 68 (16%)
Their total value: 2,915.66
DRAFT: 2

Line items attached to excluded invoices: 13 of 77


### 6.4 Date coverage

Determines whether any time-series analysis is defensible.

In [26]:
inv["_date"] = pd.to_datetime(inv["DateString"], errors="coerce")
inv["_due"] = pd.to_datetime(inv["DueDateString"], errors="coerce")
today = pd.Timestamp.today().normalize()

print(f"Invoice date range: {inv['_date'].min().date()} "
      f"-> {inv['_date'].max().date()}")
print(f"Today: {today.date()}")
print(f"Future-dated invoices: {(inv['_date'] > today).sum()} of {len(inv)}")
print(f"Distinct months: {inv['_date'].dt.to_period('M').nunique()}\n")
print(inv["_date"].dt.to_period("M").value_counts().sort_index())

Invoice date range: 2026-06-06 -> 2026-10-18
Today: 2026-08-10
Future-dated invoices: 11 of 68
Distinct months: 5

_date
2026-06    26
2026-07    16
2026-08    16
2026-09     5
2026-10     5
Freq: M, Name: count, dtype: int64


**Finding — date coverage**

| | |
|---|---|
| Full range | 2026-06-06 → 2026-10-18 |
| Today | 2026-08-10 |
| Future-dated | **11 of 68** |
| Distinct months | 5 (3 usable) |

Monthly distribution: Jun 26 · Jul 16 · Aug 16 · Sep 5 · Oct 5

**Usable historical window: June–August 2026 — three months, 57 invoices.**

**Why this happens.** The Xero Demo Company generates data on a rolling window
relative to the current date, which is why a third of the invoices are dated
in the future. These are not forecasts or a planning pipeline; they are an
artifact of demo data generation.

**Implications**

1. **No trend, seasonality, or cohort analysis is defensible.** Three months of
   data across 57 invoices cannot support a claim about direction.
2. **Any time-series visual must apply an as-of filter** (`invoice_date <=
   current_date`) or it will plot activity that has not occurred.
3. **The source is not stable.** The demo company regenerates, so re-running
   this extraction in a month yields a different date window and different
   record IDs. **Raw JSON must be landed in BigQuery and treated as the source
   of truth**, rather than assuming the API can reproduce today's dataset.

**Related — `UpdatedDateUTC` is unusable here.** Sample values fall in 2008
while transaction dates are 2026. Incremental extraction via
`If-Modified-Since` can be implemented, but **cannot be validated against this
source.**

---
## 7. Findings Summary

### Confirmed

| # | Finding | Evidence |
|---|---|---|
| 1 | All six entities have valid primary keys | Unique + non-null, 6/6 (§5.1) |
| 2 | Full referential integrity — no orphans, no null FKs | 8/8 relationships clean (§5.2) |
| 3 | Tax semantics: Exclusive → `SubTotal`, Inclusive → `Total` | 49/49 and 19/19, no exceptions (§6.1) |
| 4 | Proposed net-amount normalisation reconciles to header | 68/68, zero failures (§6.1) |
| 5 | Single currency throughout | USD on all 68 invoices; `CurrencyRate` unused |
| 6 | Three 1:M relationships, all low fan-out | Max 3 lines/invoice, 6 invoices/contact (§5.3) |

### Data traps caught

| # | Trap | Impact if missed | Fix (verified) |
|---|---|---|---|
| 1 | API returns soft-deleted payments | Cash overstated ~45% | `status = 'AUTHORISED'` → mismatches 17 → 0 |
| 2 | `LineAmount` means different things per invoice | Revenue overstated on 19/68 | Normalise Inclusive by subtracting tax |
| 3 | Same soft-delete pattern on invoices and bank txns | Cancelled txns counted as real | Status filter on all transactional models |
| 4 | `Balances.*` on contacts are API-computed snapshots | Stale figures contradicting the facts | Exclude from `dim_contact`; derive in marts |
| 5 | Contact `Name` not unique (49 of 50) | Wrong grouping on name | Join on `ContactID` only |

### Constraints on analysis

| # | Constraint | Implication |
|---|---|---|
| 1 | 57 live invoices, 64 live lines, 34 live payments | No statistically meaningful findings |
| 2 | Usable window is 3 months (Jun–Aug 2026) | No trend, seasonality or cohort analysis |
| 3 | 11 invoices future-dated | Time-series visuals need an as-of filter |
| 4 | 1.13 lines per invoice | Product/line-mix analysis is near-empty |
| 5 | 17 of 58 accounts transacted | Category breakdown has 17 buckets |
| 6 | 29 of 50 contacts have invoices | 21 dimension rows with no facts |
| 7 | All payments hit one bank account | `payment → account` supports no breakdown |
| 8 | `Item` populated on 39% of lines | `dim_item` not viable in v1 |
| 9 | Demo Company regenerates; `UpdatedDateUTC` stale (2008) | Land raw in BigQuery as source of truth; incremental load implementable but not verifiable |

**Framing.** This dataset supports an **engineering** portfolio piece — OAuth
with least-privilege scopes, incremental ingestion, dimensional modelling,
tested transformations, CI/CD. It does **not** support a business-insights
narrative, and claiming one would be the weaker move. The two traps above are
the substantive analytical content: both produce plausible-looking wrong
numbers, and both are caught by tests that fail on raw and pass on modelled
data.

### Must be handled in staging

| # | Issue | Handling | dbt test |
|---|---|---|---|
| 1 | Deleted payments | `where status = 'AUTHORISED'` | `invoice_id` unique in `fact_payments` |
| 2 | Voided/deleted invoices | Exclude `VOIDED`, `DELETED` (11 invoices, 13 lines) | `accepted_values` on status |
| 3 | Draft invoices | **Keep with flag** — pending, not cancelled | Excluded from revenue measures |
| 4 | Mixed tax basis | `line_amount_net` / `line_amount_gross` | `sum(net) = subtotal` per invoice |
| 5 | Epoch-millis dates | Parse `*String` (ISO-8601) columns only | `not_null` on `date_key` |
| 6 | Denormalised `Contact.*` / `Invoice.*` | Drop; source dimensions from their own endpoints | `relationships` tests on FKs |
| 7 | Contact balance snapshots | Exclude; derive in marts | — |

---
## 8. Proposed Star Schema

### Grain statements

- **`fact_invoice_lines`** — one row per invoice line item on a live
  (non-voided, non-deleted) invoice. **64 rows.**
- **`fact_payments`** — one row per authorised payment. **34 rows.**

### Model

```
fact_invoice_lines                          64 rows
  PK    invoice_line_key         <- LineItemID
  FK    invoice_id               <- InvoiceID
        contact_key              <- Contact.ContactID
        account_key              <- AccountID
        date_key                 <- DateString (invoice date)
  deg   invoice_number, invoice_type (ACCREC/ACCPAY),
        invoice_status, line_amount_types, description, tax_type
  M     quantity, unit_amount,
        line_amount_net, line_amount_gross, tax_amount

fact_payments                               34 rows
  PK    payment_key              <- PaymentID
  FK    invoice_id               <- Invoice.InvoiceID   (1:1 after filter)
        account_key              <- Account.AccountID   (constant, kept for completeness)
        date_key                 <- DateString (payment date)
  deg   payment_type (ACCRECPAYMENT/ACCPAYPAYMENT), is_reconciled,
        batch_payment_id
  M     amount, bank_amount

dim_contact                                 50 rows
  PK    contact_key              <- ContactID
        name, is_customer, is_supplier, email_address,
        first_name, last_name
  NOTE  Balances.* excluded — API snapshots, derived in marts instead

dim_account                                 58 rows
  PK    account_key              <- AccountID
        code, name, type, class, description,
        reporting_code, reporting_code_name, tax_type, system_account
  NOTE  Class drives the P&L / balance-sheet split (EXPENSE 29,
        LIABILITY 15, ASSET 9, REVENUE 3, EQUITY 2)

dim_date                                    generated
  PK    date_key
        date, year, quarter, month, month_name, week,
        day_of_week, is_weekend, is_future (as-of guard)
  NOTE  Covers 2026-06-01 -> 2026-10-31 to span observed range
```

### Decisions and rationale

**Single `fact_invoice_lines` with an `invoice_type` flag, not split AR/AP.**
ACCREC and ACCPAY share an identical schema; splitting would duplicate every
staging model and test for no gain. The flag is mandatory in every mart-level
filter, since summing across both nets receivables against payables. Volume
also argues against splitting — 27 sales invoices does not warrant its own
pipeline.

**VOIDED and DELETED filtered in staging, not flagged in marts.**
These are not transactions that happened. Retaining them invites accidental
inclusion. The count (11 invoices, $2,915.66) is documented here rather than
carried into the model. Note all voids/deletes are on the AP side — no sales
invoice was voided.

**DRAFT kept with a status flag.**
Unlike voided, a draft is a pending real transaction. Keeping the 2 draft
invoices allows a pipeline-value measure while excluding them from revenue.

**No unknown members in dimensions.**
Justified by §5.2 — zero null FKs and zero orphans across all eight
relationships. Would be required in a production source.

**No `dim_item` in v1.**
`ItemCode` covers 30 of 77 lines (39%). `dim_account` gives 100% coverage and
serves the same categorisation purpose.

**Bank transactions deferred.**
14 live, single account, all `SPEND`, line items unfetched, and a risk of
double-counting cash already represented in `fact_payments`.

**Credit notes deferred.**
5 records, `AmountCredited` on 5 invoices. Real contra-revenue, but too thin to
model in v1. Documented as a known omission — invoice net value will slightly
overstate where credits exist.

### Raw layer

Land the unmodified JSON payload per entity with an ingestion timestamp, and
transform in dbt rather than in Python. Two reasons:

1. **Testability.** Transformation logic lives in version control with tests
   attached, not in an untested extraction script.
2. **Immutability.** The Demo Company regenerates on a rolling window (§6.4).
   Once landed, the raw layer is a stable historical record even when the API
   no longer returns the same data.

### Open items for the next session

- BigQuery dataset + raw table design (JSON column vs. typed columns)
- Incremental extraction via `If-Modified-Since` — implementable, but
  unverifiable against this source given the stale `UpdatedDateUTC`
- Pagination loop — not exercised at this volume, but required for correctness
- `dbt test` in GitHub Actions on every push